# TARGET EDA (streaming, memory-efficient)

This notebook performs quick EDA on `application_train.csv` using chunked reads so other CSVs are never loaded. Outputs (CSV reports and figures) are written into `outputs/reports/` and `outputs/figures/`.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Paths
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_ROOT / 'data' / 'raw'
FILE = RAW / 'application_train.csv'
OUT_REPORT = PROJECT_ROOT / 'outputs' / 'reports'
OUT_FIG = PROJECT_ROOT / 'outputs' / 'figures'
OUT_REPORT.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

# Chunk size chosen to be small enough for 16GB machines and fast IO
CHUNK = 50000

# Columns of interest
CAT_COLS = [
    'NAME_CONTRACT_TYPE',
    'CODE_GENDER',
    'FLAG_OWN_CAR',
    'FLAG_OWN_REALTY',
    'NAME_INCOME_TYPE',
]
NUM_COLS = ['AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'DAYS_BIRTH', 'DAYS_EMPLOYED']

print('File to analyze:', FILE)
print('Chunk size:', CHUNK)

File to analyze: E:\ifrs9-home-credit-pd-model\data\raw\application_train.csv
Chunk size: 50000


In [2]:
# PASS 1: totals, missing counts, target distribution, categorical aggregates, numeric min/max
total_rows = 0
target_counts = {0: 0, 1: 0}
missing_counts = None
numeric_min = {c: np.inf for c in NUM_COLS}
numeric_max = {c: -np.inf for c in NUM_COLS}
cat_stats = {c: {} for c in CAT_COLS}  # {col: {category: [count, defaults]}}

reader = pd.read_csv(FILE, encoding='latin1', chunksize=CHUNK)
for i, chunk in enumerate(reader):
    # update totals
    n = len(chunk)
    total_rows += n
    # TARGET counts (ensure int)
    if 'TARGET' in chunk.columns:
        target_counts[0] += int((chunk['TARGET'] == 0).sum())
        target_counts[1] += int((chunk['TARGET'] == 1).sum())
    # missing counts
    if missing_counts is None:
        missing_counts = chunk.isna().sum().to_dict()
    else:
        miss = chunk.isna().sum().to_dict()
        for k, v in miss.items():
            missing_counts[k] = missing_counts.get(k, 0) + int(v)
    # numeric min/max
    for c in NUM_COLS:
        if c in chunk.columns:
            col = chunk[c].dropna()
            if not col.empty:
                mn = col.min()
                mx = col.max()
                if pd.notna(mn) and mn < numeric_min[c]:
                    numeric_min[c] = float(mn)
                if pd.notna(mx) and mx > numeric_max[c]:
                    numeric_max[c] = float(mx)
    # categorical aggregates
    for c in CAT_COLS:
        if c in chunk.columns:
            grp = chunk.groupby(c)['TARGET'].agg(['count', 'sum']).reset_index()
            for _, r in grp.iterrows():
                key = r[c] if pd.notna(r[c]) else '__MISSING__'
                st = cat_stats[c].get(key, [0, 0])
                st[0] += int(r['count'])
                st[1] += int(r['sum'])
                cat_stats[c][key] = st
    if (i + 1) % 10 == 0:
        print(f'Processed {total_rows} rows...')

print('Pass 1 done. Rows:', total_rows)
print('TARGET counts:', target_counts)

Pass 1 done. Rows: 307511
TARGET counts: {0: 282686, 1: 24825}


In [3]:
# Save basic shape and target counts
shape_report = pd.DataFrame([{'file': str(FILE), 'rows': total_rows}])
shape_report.to_csv(OUT_REPORT / 'shape_and_rows.csv', index=False)

tc = pd.DataFrame([{'label': '0', 'count': target_counts[0]}, {'label': '1', 'count': target_counts[1]}])
tc['default_rate'] = tc.apply(lambda r: r['count'] / total_rows if total_rows else np.nan, axis=1)
tc.to_csv(OUT_REPORT / 'target_counts.csv', index=False)

print(tc)
print('\nDefault rate (1s):', target_counts[1] / total_rows)

# Missing percentages
miss_df = pd.DataFrame([{'column': k, 'missing_count': v, 'missing_pct': 100.0 * v / total_rows} for k, v in missing_counts.items()])
miss_df = miss_df.sort_values('missing_pct', ascending=False).reset_index(drop=True)
miss_df.to_csv(OUT_REPORT / 'missing_percentages_all.csv', index=False)
miss_df.head(20).to_csv(OUT_REPORT / 'missing_top20.csv', index=False)
print('\nTop 20 columns by missing percentage:')
print(miss_df.head(20))

  label   count  default_rate
0     0  282686      0.919271
1     1   24825      0.080729

Default rate (1s): 0.08072881945686496

Top 20 columns by missing percentage:
                      column  missing_count  missing_pct
0             COMMONAREA_AVG         214865    69.872297
1            COMMONAREA_MODE         214865    69.872297
2            COMMONAREA_MEDI         214865    69.872297
3   NONLIVINGAPARTMENTS_MEDI         213514    69.432963
4   NONLIVINGAPARTMENTS_MODE         213514    69.432963
5    NONLIVINGAPARTMENTS_AVG         213514    69.432963
6         FONDKAPREMONT_MODE         210295    68.386172
7       LIVINGAPARTMENTS_AVG         210199    68.354953
8      LIVINGAPARTMENTS_MEDI         210199    68.354953
9      LIVINGAPARTMENTS_MODE         210199    68.354953
10            FLOORSMIN_MODE         208642    67.848630
11             FLOORSMIN_AVG         208642    67.848630
12            FLOORSMIN_MEDI         208642    67.848630
13           YEARS_BUILD_AVG     

In [4]:
# Categorical default rates (from pass 1)
cat_rows = []
for c, d in cat_stats.items():
    for k, (cnt, defaults) in d.items():
        cat_rows.append({'column': c, 'category': k, 'count': cnt, 'defaults': defaults, 'default_rate': defaults / cnt if cnt else np.nan})
cat_df = pd.DataFrame(cat_rows)
cat_df.to_csv(OUT_REPORT / 'categorical_default_rates.csv', index=False)
print('Categorical default rates saved:', OUT_REPORT / 'categorical_default_rates.csv')

# Plot categorical default rates for each requested column
for c in CAT_COLS:
    if c not in cat_df['column'].unique():
        continue
    tmp = cat_df[cat_df['column'] == c].sort_values('default_rate', ascending=False)
    plt.figure(figsize=(8, 4))
    sns.barplot(data=tmp, x='default_rate', y='category')
    plt.title(f'Default rate by {c}')
    plt.xlabel('Default rate')
    plt.tight_layout()
    fname = OUT_FIG / f'default_rate_{c}.png'
    plt.savefig(fname, dpi=150)
    plt.close()
    print('Saved figure', fname)

# Missing top20 plot
plt.figure(figsize=(8, 6))
tmpm = miss_df.head(20).sort_values('missing_pct')
sns.barplot(x='missing_pct', y='column', data=tmpm, palette='viridis')
plt.xlabel('Missing %')
plt.title('Top 20 columns by missing %')
plt.tight_layout()
plt.savefig(OUT_FIG / 'missing_top20.png', dpi=150)
plt.close()
print('Saved missing columns figure')

Categorical default rates saved: E:\ifrs9-home-credit-pd-model\outputs\reports\categorical_default_rates.csv
Saved figure E:\ifrs9-home-credit-pd-model\outputs\figures\default_rate_NAME_CONTRACT_TYPE.png


Saved figure E:\ifrs9-home-credit-pd-model\outputs\figures\default_rate_CODE_GENDER.png
Saved figure E:\ifrs9-home-credit-pd-model\outputs\figures\default_rate_FLAG_OWN_CAR.png


Saved figure E:\ifrs9-home-credit-pd-model\outputs\figures\default_rate_FLAG_OWN_REALTY.png
Saved figure E:\ifrs9-home-credit-pd-model\outputs\figures\default_rate_NAME_INCOME_TYPE.png


C:\Users\mypc\AppData\Local\Temp\ipykernel_51824\3074303518.py:28: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='missing_pct', y='column', data=tmpm, palette='viridis')


Saved missing columns figure


In [5]:
# PASS 2: numeric binned summaries using equal-width bins between min and max
NUM_BINS = 10
bin_edges = {}
for c in NUM_COLS:
    mn = numeric_min.get(c, np.nan)
    mx = numeric_max.get(c, np.nan)
    if not np.isfinite(mn) or not np.isfinite(mx):
        bin_edges[c] = None
    else:
        if mn == mx:
            # make a tiny range
            mn = mn - 0.5 if not np.isnan(mn) else -0.5
            mx = mx + 0.5 if not np.isnan(mx) else 0.5
        bin_edges[c] = np.linspace(mn, mx, NUM_BINS + 1)

# initialize stats holders
bin_stats = {c: {} for c in NUM_COLS}

reader = pd.read_csv(FILE, encoding='latin1', chunksize=CHUNK)
for i, chunk in enumerate(reader):
    for c in NUM_COLS:
        if c not in chunk.columns or bin_edges[c] is None:
            continue
        arr = chunk[c]
        # cut into bins; include NaNs separately
        labels = pd.cut(arr, bins=bin_edges[c], include_lowest=True).rename('bin')
        grp = chunk.groupby(labels, observed=False)['TARGET'].agg(['count', 'sum']).reset_index().dropna(subset=['bin'], how='all')
        for _, r in grp.iterrows():
            key = str(r['bin'])
            st = bin_stats[c].get(key, [0, 0])
            st[0] += int(r['count'])
            st[1] += int(r['sum'])
            bin_stats[c][key] = st
    if (i + 1) % 10 == 0:
        print(f'Pass2 processed {(i+1)*CHUNK} rows...')

# convert bin_stats to DataFrames and save
for c in NUM_COLS:
    rows = []
    for k, (cnt, defs) in bin_stats[c].items():
        rows.append({'column': c, 'bin': k, 'count': cnt, 'defaults': defs, 'default_rate': defs / cnt if cnt else np.nan})
    dfc = pd.DataFrame(rows)
    if not dfc.empty:
        dfc = dfc.sort_values('bin')
        dfc.to_csv(OUT_REPORT / f'numeric_binned_{c}.csv', index=False)
        # plot default rate by bin
        plt.figure(figsize=(8, 4))
        sns.barplot(data=dfc, x='bin', y='default_rate')
        plt.xticks(rotation=45, ha='right')
        plt.xlabel('Bin')
        plt.ylabel('Default rate')
        plt.title(f'Default rate by binned {c}')
        plt.tight_layout()
        plt.savefig(OUT_FIG / f'default_rate_binned_{c}.png', dpi=150)
        plt.close()
        print('Saved numeric binned report and figure for', c)
    else:
        print('No data for numeric column', c)

Saved numeric binned report and figure for AMT_INCOME_TOTAL
Saved numeric binned report and figure for AMT_CREDIT


Saved numeric binned report and figure for AMT_ANNUITY
Saved numeric binned report and figure for DAYS_BIRTH


Saved numeric binned report and figure for DAYS_EMPLOYED
